# Exercise: A Parameter Sweep with PyLauncher

A damped oscillator answers an earthquake-engineering question with one number. Shake it at a given frequency and, once the transient dies away, the steady response settles at some multiple of the static displacement, the **dynamic amplification**. Your task is to sweep the forcing frequency with [PyLauncher](https://github.com/TACC/pylauncher) and [dapi](https://designsafe-ci.github.io/dapi/), then reassemble the resonance curve from the archived results.

Work through the TODO cells. Each has a hint behind the fold, and the [solution notebook](pylauncher_sweep.ipynb) shows one complete answer.

![A mass on a spring and damper, forced by a sine load, beside the amplification curve the sweep reassembles, with one marked point per task.](resonance_schematic.png)

The oscillator obeys

$$\ddot{x} + 2\xi\omega_n\dot{x} + \omega_n^2 x = \frac{F_0}{m}\sin(\Omega t)$$

and each task integrates it at one forcing frequency $\Omega$, reporting the **dynamic amplification** $D$, the steady-state peak displacement over the static displacement $F_0/k$. Sweeping the frequency ratio $r = \Omega/\omega_n$ from 0.2 to 2.0 traces $D(r)$, which the closing plot compares with the closed form.

In [ ]:
%pip install dapi --quiet

**Restart the kernel once after the install**, then run from the next cell.

In [ ]:
import os
from pathlib import Path
from dapi import DSClient

ds = DSClient()

# On DesignSafe JupyterHub, work in MyData; anywhere else, work in a local
# folder, which the sweep uploads automatically at submission.
if "JUPYTER_SERVER_ROOT" in os.environ:
    work_root = Path(os.environ["JUPYTER_SERVER_ROOT"]) / "MyData"
else:
    work_root = Path.cwd()

## The task script (given)

The script integrates the oscillator at the frequency ratio it receives and writes the steady-state amplification to its own output folder. PyLauncher passes each task its `--ratio` and `--output`.

In [ ]:
input_dir = work_root / "resonance_sweep"
input_dir.mkdir(parents=True, exist_ok=True)

oscillator_script = """\
# One resonance-sweep task: integrate a damped oscillator at one
# forcing frequency and record the steady-state dynamic amplification.

import argparse
import json
import os

import numpy as np

parser = argparse.ArgumentParser()
parser.add_argument("--ratio", type=float, required=True)  # forcing / natural frequency
parser.add_argument("--output", type=str, required=True)
args = parser.parse_args()

xi = 0.05          # damping ratio
wn = 1.0           # natural frequency (rad/s)
w = args.ratio * wn

# x'' + 2 xi wn x' + wn^2 x = sin(w t), central differences
dt = 0.002
n = int(80 * 2 * np.pi / wn / dt)  # 80 natural periods, transient plus steady state
x = np.zeros(n)
t = np.arange(n) * dt
for i in range(1, n - 1):
    a = np.sin(w * t[i]) - 2 * xi * wn * (x[i] - x[i - 1]) / dt - wn**2 * x[i]
    x[i + 1] = 2 * x[i] - x[i - 1] + a * dt**2

steady = x[int(0.75 * n):]         # last quarter, transient long gone
amplification = np.max(np.abs(steady)) * wn**2   # peak dynamic / static response

os.makedirs(args.output, exist_ok=True)
json.dump(
    {"ratio": args.ratio, "amplification": amplification},
    open(os.path.join(args.output, "result.json"), "w"),
)
print(f"ratio={args.ratio}: amplification={amplification:.3f}")
"""

(input_dir / "oscillator.py").write_text(oscillator_script)
print(f"Wrote {input_dir}/oscillator.py")

## TODO 1. Define the sweep

Sweep the frequency ratio from 0.2 to 2.0 in 25 steps, and write the command template each task runs. The placeholder name in the template is replaced per task.

<details><summary>Hint</summary>

A dict with one key, `RATIO`, holding the list of values, e.g. `[round(0.2 + 0.075 * i, 3) for i in range(25)]`. The template is `"python3 oscillator.py --ratio RATIO --output out_RATIO"`; every bare `RATIO` token is substituted.
</details>

Docs: [PyLauncher Parameter Sweeps](https://designsafe-ci.github.io/dapi/examples/pylauncher).

In [ ]:
sweep = ...  # TODO
command = ...  # TODO

## TODO 2. Preview before generating

Expand the sweep without writing anything and check the count, 25 commands, one per frequency.

<details><summary>Hint</summary>

`ds.jobs.parametric_sweep.generate(command, sweep, preview=True)` returns the expanded table.
</details>

In [ ]:
...  # TODO

## TODO 3. Generate and submit

Write the PyLauncher files into the sweep folder, then submit it as one `python-s3` job on 48 cores of a `skx-dev` node and monitor to completion. A local folder uploads automatically.

<details><summary>Hint</summary>

`ds.jobs.parametric_sweep.generate(command, sweep, str(input_dir))` writes the files. Then `ds.jobs.parametric_sweep.submit(str(input_dir), app_id="python-s3", allocation=..., node_count=1, cores_per_node=48, max_minutes=15, queue="skx-dev")` and `job.monitor(interval=30)`.
</details>

Docs: [Job Monitoring](https://designsafe-ci.github.io/dapi/jobs#job-monitoring).

In [ ]:
allocation = "DS-Portal-SPARC2026"  # <-- replace with your allocation

commands = ...  # TODO: generate the sweep files
job = ...  # TODO: submit
final = ...  # TODO: monitor to completion
job.print_runtime_summary()

## TODO 4. Reassemble the resonance curve

Each task archived a `result.json` with its ratio and amplification under the job's `inputDirectory`. Gather them, sort by ratio, and plot the points over the closed form

$$D(r) = \frac{1}{\sqrt{(1 - r^2)^2 + (2 \xi r)^2}}$$

Where does the peak land, and how high is it for 5% damping?

<details><summary>Hint</summary>

`job.archive_uri + "/inputDirectory"` lists the `out_*` folders; download each `result.json` and collect `(ratio, amplification)` pairs. The peak should sit near $r = 1$ at roughly $1/(2\xi) = 10$.
</details>

Docs: [Output Management](https://designsafe-ci.github.io/dapi/jobs#output-management).

In [ ]:
import json as _json  # noqa: F401  (your answer uses it)
import numpy as np  # noqa: F401  (your answer uses it)
import matplotlib.pyplot as plt  # noqa: F401  (your answer uses it)

%matplotlib inline

results_uri = ...  # TODO
points = ...  # TODO: gather and sort (ratio, amplification) pairs
# TODO: plot the points over the closed form

## Going further

Halve the damping in `oscillator.py` and rerun; the peak should double and narrow. The [OpenSees sweep exercise](pylauncher_opensees_exercise.ipynb) runs the same machinery on a structural model.